In [ ]:
import numpy as np
from astropy.table import Table
from astropy.coordinates import SpectralCoord
import astropy.units as u
import matplotlib.pyplot as plt
from specutils import Spectrum

In [ ]:
# Aux functions
def gaussian_profile(vel_array, intensity, velocity, width):
    sigma = width / (2 * np.sqrt(2 * np.log(2)))
    return intensity * np.exp(-(vel_array - velocity)**2 / (2 * sigma**2))
    # return int * np.exp(-(vel_array - velocity)**2 / (2 * width**2))

In [ ]:
# Read SLIM data (should be only one)
molecule = "PO,v=0"
slim_freq_unit = u.MHz

slim_filename = "cross-corr-dats/slim_table_region_10.csv"
# slim_filename = "cross-corr-dats/SLIM-table-SO.csv"
slim_table = Table.read(slim_filename, format='csv', delimiter=",")
slim_table["Frequency"].unit = slim_freq_unit

In [ ]:
# Read spectra data (can be multiple files)
spectra_filenames = [
    "cross-corr-dats/01_IRAS_16293-2422_spw27.dat",
    "cross-corr-dats/01_IRAS_16293-2422_spw29.dat",
]
spectra_freq_unit = u.GHz
spectra_flux_unit = u.Jy
# Molecule filter
molecule_filter = (slim_table["Formula"] == molecule)

# Frequency filter
# Using Spectra class. Stores Spectrum anD Table just in case
spectra_tables = []
spectra = []
for filename in spectra_filenames:
    spectrum_table = Table.read(filename, format="ascii")
    spectra_tables.append(spectrum_table)
    spectra.append(
        Spectrum(
            spectral_axis=spectrum_table.columns[0].data * spectra_freq_unit,
            flux=spectrum_table.columns[1].data * spectra_flux_unit,
        )
    )
freq_range_filter = np.zeros(len(slim_table), dtype=bool)
for spectrum in spectra:
    spec_freq = spectrum.spectral_axis
    freq_range_filter |= (
        (slim_table["Frequency"] > spec_freq.min()) &
        (slim_table["Frequency"] < spec_freq.max())
    )
slim_table_mol = slim_table[molecule_filter & freq_range_filter]


# Old individual version to test
spec_obsfreq_value_old = spectra_tables[0].columns[0].data
spectrum_flux_value_old = spectra_tables[0].columns[1].data
spec_obsfreq_old = spec_obsfreq_value_old * spectra_freq_unit
molecule_filter_old = (slim_table["Formula"] == molecule)

freq_range_filter_old = ((slim_table["Frequency"] > spec_obsfreq_old.min()) &
                     (slim_table["Frequency"] < spec_obsfreq_old.max()))

slim_table_mol_old = slim_table[molecule_filter_old & freq_range_filter_old]

In [ ]:
slim_table_mol

In [ ]:
# velocity = spectrum.spectral_axis.to(
#     u.km/u.s,
#     doppler_rest=rest_freq,
#     doppler_convention="radio",
# )

# or
# velocity = spectrum.spectral_axis.to(
#     u.km/u.s,
#     equivalencies=u.doppler_radio(rest_frequency),
# )

In [ ]:
# old safe guard version!!!

group_col = "Transition" if "Transition" in slim_table_mol.colnames else "Frequency"
slim_table_mol_grouped = slim_table_mol.group_by(group_col)

for key, group in zip(slim_table_mol_grouped.groups.keys,
                      slim_table_mol_grouped.groups):
    # Here group is each transition table of the molecule

    # Rest frequency of this transition
    rest_freq = group["Frequency"].quantity[0]

    # # Check if all rows are the same transition
    # q = group["Frequency"].quantity
    # if "Transition" in group.colnames:
    #     if np.all(group["Transition"] == group["Transition"][0]):
    #         spec_obsfreq_sc = SpectralCoord(
    #             spec_obsfreq,
    #             doppler_rest=group["Frequency"].quantity[0],
    #             doppler_convention='radio',
    #         )
    # elif np.allclose(
    #     q.to_value(u.MHz), q[1].to_value(u.MHz),
    #     rtol=1e-12, atol=1e-12):  # This has precision up to the Hz on MHz arrays
    #     spec_obsfreq_sc = SpectralCoord(
    #         spec_obsfreq,
    #         doppler_rest=group["Frequency"].quantity[0],
    #         doppler_convention='radio',
    #     )
    # else: print("error")

    component_lines = np.column_stack([
        group["Intensity"],
        group["Velocity"],
        group["Width"],
    ])

    for spectrum in spectra:

        # Check if transition is inside this spectrum
        if not (
            spectrum.spectral_axis.min() <= rest_freq <= spectrum.spectral_axis.max()
        ):
            continue

        found = True

        # Create models

        # Velocity axis for this spectrum
        spec_vel_value = spectrum.spectral_axis.to(
            u.km / u.s,
            doppler_rest=rest_freq,
            doppler_convention="radio",
        ).value
        # Observed flux
        spectrum_flux_value = spectrum.flux.value
        # Create Gaussian model
        model = np.zeros_like(spec_vel_value)

        for intensity, vel, width in component_lines:
            model += gaussian_profile(spec_vel_value, intensity, vel, width)

        # Plot
        figsize = (6, 2)
        fig, ax = plt.subplots(1, 1, figsize=figsize)

        ax.plot(
            spec_vel_value,
            spectrum_flux_value,
            drawstyle="steps-mid",
            color="black",
            alpha=0.8,
        )
        ax.plot(
            spec_vel_value,
            model,
            drawstyle="steps-mid",
            color="red",
        )

        # ax.axvline(0.0, color="k", ls="--", alpha=0.5)

        ax.set_ylabel("Intensity")
        ax.set_xlabel("Velocity [km/s]")
        ax.set_title("SLIM generated model over observed spectrum")
        ax.text(
            0.02,
            0.98,
            f"{key[0]}",
            ha="left",
            va="top",
            transform=ax.transAxes,
        )

        ax.set_xlim(-40, 40)
        ax.set_ylim(-1.5 * model.max(), 2 * model.max())

        plt.tight_layout()
        # plt.savefig(f"{key[0]}-mult.png", dpi=300)
        plt.show()

    if not found:
        raise RuntimeError(
            f"Transition at {rest_freq} was not found in any selected spectrum."
        )

---

In [ ]:
# # old safe guard version!!!

# group_col = "Transition" if "Transition" in slim_table_mol.colnames else "Frequency"
# slim_table_mol_grouped = slim_table_mol.group_by(group_col)

# for key, group in zip(slim_table_mol_grouped.groups.keys,
#                       slim_table_mol_grouped.groups):
#     # Here group is each transition table of the molecule

#     q = group["Frequency"].quantity

#     # Check if all rows are the same transition
#     if "Transition" in group.colnames:
#         if np.all(group["Transition"] == group["Transition"][0]):
#             spec_obsfreq_sc = SpectralCoord(
#                 spec_obsfreq,
#                 doppler_rest=group["Frequency"].quantity[0],
#                 doppler_convention='radio',
#             )
#     elif np.allclose(
#         q.to_value(u.MHz), q[1].to_value(u.MHz),
#         rtol=1e-12, atol=1e-12):  # This has precision up to the Hz on MHz arrays
#         spec_obsfreq_sc = SpectralCoord(
#             spec_obsfreq,
#             doppler_rest=group["Frequency"].quantity[0],
#             doppler_convention='radio',
#         )
#     else: print("error")

#     component_lines = np.column_stack([
#         group["Intensity"],
#         group["Velocity"],
#         group["Width"],
#     ])

#     # Create models
#     spec_vel_value = spec_obsfreq_sc.to(u.km / u.s).value
#     model = np.zeros_like(spec_vel_value)

#     for intensity, vel, width in component_lines:
#         model += gaussian_profile(spec_vel_value, intensity, vel, width)

#     # Results 

#     figsize=(6,2)
#     fig, ax = plt.subplots(1, 1, figsize=figsize)

#     ax.plot(spec_vel_value, spectrum_flux_value, drawstyle='steps-mid', 
#                 color="black", alpha=0.8)
#     ax.plot(spec_vel_value, model, drawstyle='steps-mid', color="red")
#     # ax.axvline(0.0, color="k", ls="--", alpha=0.5)
#     ax.set_ylabel("Intensity")
#     ax.set_title("SLIM generated model over observed spectrum")
#     ax.set_xlabel("Velocity [km/s]")
#     ax.text(0.02, 0.98, f"{key[0]}", ha='left', va='top', transform=ax.transAxes)

#     ax.set_xlim(-40, 40)
#     ax.set_ylim(-8/10*model.max(), 1.5 * model.max())

#     plt.tight_layout()
#     # plt.savefig(f"{key[0]}-one.png", dpi=300)
#     plt.show()

---

---

---

# correlation part

In [ ]:
# # Compute cross-correlation


# ## Numpy cross-correlation
# corr_np = np.correlate(simulated_data, model, mode='same')

# # Normalize correlation to S/N units
# corr_np /= np.std(corr_np)
# # corr_np /= (np.std(corr_np)**2)

# # Compute lag axis
# lags = (np.arange(n_chan) - n_chan//2) * dv


# ## Scipy FFT cross-correlation

# F_data = np.fft.fft(simulated_data)
# F_model = np.fft.fft(model)
# corr_fft = np.fft.ifft(F_data * np.conj(F_model)).real
# # Move zero lag to the center
# corr_fft = np.fft.fftshift(corr_fft)
# corr_fft /= np.std(corr_fft)

In [ ]:
# # Results 

# fig, axes = plt.subplots(4, 1, figsize=figsize, sharex=True)

# axes[0].plot(vel, model, color="red")
# axes[0].set_ylabel("Model")
# axes[0].set_title("Model molecular emission")

# axes[1].plot(vel, simulated_data, label="Simulated data",
#              color="black", alpha=0.8)
# axes[1].plot(vel, model, color="red", label="True model")
# # axes[1].axvline(0.0, color="k", ls="--", alpha=0.5)
# axes[1].set_ylabel("Intensity")
# axes[1].set_title("Noisy spectrum with true signal location")
# axes[1].set_xlabel("Velocity [km/s]")

# # Old plotting velocity and corr (incorrect, I needed to
# # compute the lag axis)
# # axes[2].plot(vel, corr_np)
# axes[2].plot(lags, corr_np)
# axes[2].set_ylabel("Correlation (S/N)")
# axes[2].set_title("Numpy Cross-correlation")

# axes[3].plot(lags, corr_fft)
# axes[3].set_xlabel("Velocity shift (km/s)")
# axes[3].set_ylabel("Correlation")
# axes[3].set_title("Scipy FFT Cross-correlation")

# peak_sn = np.max(corr_np)
# peak_shift = lags[np.argmax(corr_np)]
# peak_vel = peak_shift + first_vel

# print(f"Peak correlation S/N            = {peak_sn:.2f}")
# print(f"Recovered shift                 = {peak_shift:.3f} km/s")
# print(f"Detected velocity of first comp = {peak_vel:.3f} km/s")
# print(f"True velocity of signal         = {true_vel:.3f} km/s")

# plt.tight_layout()
# plt.show()